# 0. Preparation

In [1]:
# if you have errors run the command pip install -r requirements.txt
import os
import json
import pandas as pd
import numpy as np
import nltk
import gensim
import ast
import re
import string
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.tokenize import word_tokenize
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/margaritacrespo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
data_path = os.path.join("..", "..", "data", "processed_dataset_v2.csv")

df = pd.read_csv(
    data_path,
    engine="python",
    on_bad_lines="skip"
)
df.head()

,pid,url,processed_text,title,description,brand_facet,category_facet,subcategory_facet,seller_facet,discount,selling_price,actual_price,average_rating,attributes
0,TKPFCZ9EA7H5FYZH,https://www.flipkart.com/yorker-solid-men-mult...,"['solid', 'women', 'multicolor', 'track', 'pan...",Solid Women Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,69.0,921.0,2999.0,3.9,"['elast', 'side', 'pocket', 'cotton', 'blend',..."
1,TKPFCZ9EJZV2UVRZ,https://www.flipkart.com/yorker-solid-men-blue...,"['solid', 'men', 'blue', 'track', 'pant', 'yor...",Solid Men Blue Track Pants,Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,66.0,499.0,1499.0,3.9,"['drawstr', 'elast', 'side', 'pocket', 'cotton..."
2,TKPFCZ9EHFCY5Z4Y,https://www.flipkart.com/yorker-solid-men-mult...,"['solid', 'men', 'multicolor', 'track', 'pant'...",Solid Men Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,68.0,931.0,2999.0,3.9,"['elast', 'side', 'pocket', 'cotton', 'blend',..."
3,TKPFCZ9ESZZ7YWEF,https://www.flipkart.com/yorker-solid-men-mult...,"['solid', 'women', 'multicolor', 'track', 'pan...",Solid Women Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,69.0,911.0,2999.0,3.9,"['elast', 'side', 'pocket', 'cotton', 'blend',..."
4,TKPFCZ9EVXKBSUD7,https://www.flipkart.com/yorker-solid-men-brow...,"['solid', 'women', 'brown', 'gray', 'track', '...","Solid Women Brown, Grey Track Pants",Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,68.0,943.0,2999.0,3.9,"['drawstr', 'elast', 'side', 'pocket', 'cotton..."


In [3]:
# Function to convert string representation of list to actual list
def safe_literal_eval(val):
    if pd.isna(val):
        return []  # Empty list for NaN values
    try:
        # ast.literal_eval to convert "['a', 'b']" to ['a', 'b']
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return [] # Return empty list if conversion fails

print("Converting columns 'processed_text' and 'attributes' to lists...")
df['processed_text'] = df['processed_text'].apply(safe_literal_eval)
df['attributes'] = df['attributes'].apply(safe_literal_eval)

# Check the conversion
print(f"The type of 'processed_text' is now: {type(df.iloc[0]['processed_text'])}")
print(f"Example of 'processed_text': {df.iloc[0]['processed_text'][:5]}...")

Converting columns 'processed_text' and 'attributes' to lists...
The type of 'processed_text' is now: <class 'list'>
Example of 'processed_text': ['solid', 'women', 'multicolor', 'track', 'pant']...


In [4]:
# we need all the tokens in a single column for the inverted index, hence we concatenate the processed_text and attributes columns
df['tokens'] = df['processed_text'] + df['attributes']

# 1. Ranking

## 1.1 TF-IDF + cosine similarity

Since we already implemented this ranking method in part 2, we will reuse the code in order to later compare with BM25 and our own ranking method. 

In [5]:
with open(os.path.join("..", "..", "project_progress", "part_2", "inverted_index.json"), "r") as f:
    inverted_index = json.load(f)

with open(os.path.join("..", "..", "project_progress", "part_2", "idf_scores.json"), "r") as f:
    idf_scores = json.load(f)

In [6]:
def setup_preprocessing_tools():
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))
    stop_words.update(['made', 'wear', 'comfort', 'quality', 
                       'look', 'perfect', 'style', 'great', 'cool'])
    stop_words.discard('no')
    stop_words.discard('not')
    return stemmer, stop_words

In [7]:
def preprocess_query(text, stemmer, stop_words):
    """
    Preprocess natural text:
    - lowercase
    - remove punctuation/numbers
    - tokenize
    - remove stopwords and non-alphabetic tokens
    - stem 
    """
    if not isinstance(text, str):
        return []
    
    text = text.replace('-', ' ')

    # lowercase
    text = text.lower()

    # remove punctuation and digits
    text = re.sub(f"[{re.escape(string.punctuation)}0-9]", " ", text)

    # tokenize
    tokens = word_tokenize(text)

    # filter tokens (stopwords, non-alpha, short tokens)
    tokens = [w for w in tokens if w.isalpha() and w not in stop_words]

    # stem
    tokens = [stemmer.stem(w) for w in tokens]

    # normalize color terms 
    color_map = {'navy': 'blue', 'grey': 'gray', 'fucsia': 'pink', 'burgundy': 'red', 'violet': 'purple', 'beige': 'brown', 'magenta': 'pink', 'indigo': 'blue', 
                 'charcoal': 'gray', 'crimson': 'red', 'teal': 'green', 'lavender': 'purple', 'mustard': 'yellow', 'turquoise': 'blue', 'peach': 'orange'}

    tokens = [color_map.get(w, w) for w in tokens]

    return tokens

In [8]:
def calculate_log_tf(tokens):
    """Calculates log-normalized TF for each term in a document's token list."""
    counts = Counter(tokens)
    return {term: 1 + np.log(count) for term, count in counts.items()} #Don't need to handle log(0) since count is always >=1 because it's only done for terms that appear in tokens

# Apply this function to every document's tokens
df['tf_scores'] = df['tokens'].apply(calculate_log_tf)

# Check the TF scores for the first document
print("TF scores for first document:")
print(df.iloc[0]['tf_scores'])

TF scores for first document:
{'solid': np.float64(1.6931471805599454), 'women': np.float64(1.0), 'multicolor': np.float64(1.6931471805599454), 'track': np.float64(1.0), 'pant': np.float64(1.0), 'yorker': np.float64(1.0), 'trackpant': np.float64(1.0), 'rich': np.float64(1.6931471805599454), 'comb': np.float64(1.0), 'cotton': np.float64(1.6931471805599454), 'give': np.float64(1.0), 'design': np.float64(1.0), 'skin': np.float64(1.0), 'friendli': np.float64(1.0), 'fabric': np.float64(1.0), 'itch': np.float64(1.0), 'free': np.float64(1.0), 'waistband': np.float64(1.0), 'year': np.float64(1.0), 'round': np.float64(1.0), 'use': np.float64(1.0), 'proudli': np.float64(1.0), 'india': np.float64(1.0), 'elast': np.float64(1.0), 'side': np.float64(1.0), 'pocket': np.float64(1.0), 'blend': np.float64(1.0)}


In [9]:
def calculate_tfidf_L2_norm(tf_scores, idf_scores_global):
    """
    Calculates the TF-IDF vector (as a dict) and the L2-norm (length)
    of that vector for a single document.
    """
    tfidf_vector = {}
    sum_of_squares = 0.0
    
    for term, tf in tf_scores.items():
        # Only include terms that are in our global IDF dictionary
        if term in idf_scores_global:
            tfidf = tf * idf_scores_global[term] # we multiply each TF value by the global IDF score
            tfidf_vector[term] = tfidf
            sum_of_squares += tfidf**2 
            
    doc_length = np.sqrt(sum_of_squares) # we compute the L2 norm 
    return tfidf_vector, doc_length

# Apply the function to the 'tf_scores' column
# This returns a tuple (tfidf_vector, doc_length), so we split it into two new columns
tfidf_results = df['tf_scores'].apply(lambda tf: calculate_tfidf_L2_norm(tf, idf_scores))
df['tfidf_vector'] = tfidf_results.apply(lambda x: x[0]) # tfidf vector dictionary
df['doc_length'] = tfidf_results.apply(lambda x: x[1]) # document length

# Check the results for the first document
print("TF-IDF vector for first document:")
print(df.iloc[0]['tfidf_vector'])
print("\nDocument length for first document:")
print(df.iloc[0]['doc_length'])

TF-IDF vector for first document:
{'solid': np.float64(1.439853988827382), 'women': np.float64(0.7098818692235951), 'multicolor': np.float64(2.979595529588283), 'track': np.float64(2.738972111440796), 'pant': np.float64(2.6079921803942043), 'yorker': np.float64(7.107318642210598), 'trackpant': np.float64(4.362279871739047), 'rich': np.float64(7.193930820733286), 'comb': np.float64(4.251348311031766), 'cotton': np.float64(0.5814838578355617), 'give': np.float64(2.490477694837456), 'design': np.float64(1.6255933526563855), 'skin': np.float64(2.881437429162399), 'friendli': np.float64(4.449799249755603), 'fabric': np.float64(1.5888186252313647), 'itch': np.float64(6.115678473094656), 'free': np.float64(3.341075651483173), 'waistband': np.float64(3.9586786970689456), 'year': np.float64(3.9775116454020374), 'round': np.float64(1.07841171810501), 'use': np.float64(2.216642663193321), 'proudli': np.float64(4.446755107374376), 'india': np.float64(0.8549122464713206), 'elast': np.float64(2.9045

In [10]:
stemmer, stop_words = setup_preprocessing_tools()
df_indexed = df.set_index('pid')

def search_tfidf(query_text, inverted_index, idf_scores, df_docs, k=10):
    """
    Performs a ranked TF-IDF search for a given query.
    
    Args:
        query_text (str): The raw query string.
        inverted_index (dict): The inverted index.
        idf_scores (dict): The pre-calculated IDF scores for all terms.
        df_docs (pd.DataFrame): The DataFrame (indexed by 'pid') 
                                containing 'tfidf_vector' and 'doc_length'.
        k (int): The number of top results to return.

    Returns:
        list: A list of score, pid, title, brand, etc. tuples, sorted by score.
    """
    
    # Preprocess the query
    query_tokens = preprocess_query(query_text, stemmer, stop_words)
    
    # Find matching documents (Conjunctive/AND query)
    try:
        # Retrieve the set of pids for each query token
        doc_sets = [set(inverted_index[token]) for token in query_tokens if token in inverted_index]
        
        # If any token is not in the index, no docs will match an AND query
        if len(doc_sets) != len(query_tokens):
            print("One or more query terms not in index. No results.")
            return []

        # Find the intersection of all sets
        matching_pids = set.intersection(*doc_sets)
    
    except KeyError:
        # This handles if a token isn't in the index, though the check above is safer
        print("Query term not in index. No results.")
        return []

    if not matching_pids:
        print("No documents contain all query terms.")
        return []

    # Calculate the Query TF-IDF Vector and its length
    query_tf = calculate_log_tf(query_tokens)
    query_tfidf_vector = {}
    query_sum_of_squares = 0.0
    
    for term, tf in query_tf.items():
        if term in idf_scores:
            tfidf = tf * idf_scores[term]
            query_tfidf_vector[term] = tfidf
            query_sum_of_squares += tfidf**2
            
    query_length = np.sqrt(query_sum_of_squares)
    
    if query_length == 0:
        print("Query vector has no length (all terms unknown).")
        return []

    # Calculate Cosine Similarity for all matching documents
    scores = {}
    
    # Filter the main DataFrame to only the documents that matched
    # .loc is fast because we set the index to 'pid'
    matching_docs = df_docs.loc[list(matching_pids)]
    
    for pid, row in matching_docs.iterrows():
        doc_tfidf_vector = row['tfidf_vector']
        doc_length = row['doc_length']
        
        # Calculate Dot Product
        dot_product = 0.0
        # Iterate over the query vector, which is much smaller
        for term, query_tfidf_val in query_tfidf_vector.items():
            if term in doc_tfidf_vector:
                dot_product += query_tfidf_val * doc_tfidf_vector[term]
        
        # Calculate Cosine Similarity
        if doc_length > 0:
            scores[pid] = dot_product / (query_length * doc_length)
    
    # Sort and return the top K results
    sorted_results = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    
    final_results = []
    # Loop through the top k sorted (pid, score) tuples
    for pid, score in sorted_results[:k]:
        
        # Get the product data
        product_data = df_docs.loc[pid]
        
        # Append a dictionary with all the info we want
        final_results.append({
            'title': product_data['title'],
            'score': score,
            'pid': pid,
            'url': product_data['url'],
            'selling_price': product_data['selling_price'],
            'brand': product_data['brand_facet']
        })
        
    return final_results

## 1.2 BM25

In [11]:
# Compute document lengths and average length
df['L_d'] = df['tokens'].apply(len)
L_ave = df['doc_length'].mean()
N = len(df)
print(f"Number of documents: {N}, Average length: {L_ave:.2f}")

Number of documents: 28080, Average length: 18.88


In [12]:
df_counts = {term: len(postings) for term, postings in inverted_index.items()}

def bm25_idf(df_t, N):
    return np.log((N - df_t + 0.5) / (df_t + 0.5) + 1)

bm25_idf_scores = {t: bm25_idf(df_t, N) for t, df_t in df_counts.items()}


In [13]:
def search_bm25(query_text, inverted_index, bm25_idf, df_docs,
                k=10, k1=1.5, b=0.75):
    """
    BM25 ranking for a given query.

    Args:
        query_text (str): Raw query string.
        inverted_index (dict): Inverted index (term -> list of pids).
        bm25_idf (dict): Precomputed BM25 IDF scores.
        df_docs (pd.DataFrame): DataFrame indexed by 'pid', containing 'tokens' and 'doc_length'.
        k (int): Number of top results to return.
        k1, b (float): BM25 hyperparameters.

    Returns:
        List[dict]: Ranked list of top-k results with pid, title, brand, etc.
    """

    # 1) Preprocess query 
    query_tokens = preprocess_query(query_text, stemmer, stop_words)
    if not query_tokens:
        return []

    # 2) Candidate documents: union of postings of query terms 
    doc_sets = [set(inverted_index[t]) for t in query_tokens if t in inverted_index]
    if not doc_sets or len(doc_sets) < len(query_tokens):
        return []
    candidate_pids = set.intersection(*doc_sets)
    if not candidate_pids:
        return []

    # 3) Compute BM25 scores for candidate documents
    scores = {}
    L_ave = df_docs['doc_length'].mean()

    for pid in candidate_pids:
        row = df_docs.loc[pid]
        doc_tokens = row['tokens']
        Ld = row['doc_length']
        score = 0.0

        # term frequencies in this doc
        tf_doc = Counter(doc_tokens)

        for t in query_tokens:
            if t not in bm25_idf_scores:
                continue
            tf_td = tf_doc.get(t, 0)
            if tf_td == 0:
                continue

            idf = bm25_idf_scores[t]

            # BM25 term contribution
            denom = tf_td + k1 * ((1 - b) + b * (Ld / L_ave))
            term_score = idf * ((k1 + 1) * tf_td / denom)
            score += term_score

        if score > 0:
            scores[pid] = score

    if not scores:
        return []

    # --- Sort and format top-k results ---
    sorted_pids = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    results = []
    for pid, s in sorted_pids:
        row = df_docs.loc[pid]
        results.append({
            "title": row.get("title", ""),
            "score": float(s),
            "pid": pid,
            "brand": row.get("brand_facet", ""),
            "url": row.get("url", ""),
            "selling_price": row.get("selling_price", np.nan)
        })
    return results


## 1.3 Our Ranking Algorithm

In [14]:
# Pre computation of average rating per brand
# We use .fillna(2.5) to give a neutral rating to products without a rating
brand_avg_ratings = df.groupby('brand_facet')['average_rating'].mean().fillna(2.5).to_dict()

DEFAULT_BRAND_RATING = 2.5 

print(f"Computed average ratings for {len(brand_avg_ratings)} brands.")

Computed average ratings for 322 brands.


In [15]:
def search_custom(query_text, inverted_index, bm25_idf, df_docs, brand_ratings_map,
                        k=10, k1=1.5, b=0.75, 
                        w_rating=0.1, w_discount=0.05, w_price = 0.1, w_brand=0.05): #w_rating, w_discount, w_price, and w_brand are new hyperparameters. Can be tuned manually
    """
    Custom ranking: BM25 boosted by average_rating and discount. The idea is to use BM25 as a base score but adding the influence of
    average_rating, discount and price as multiplicative factors so that higher rated, more discounted products and cheaper products rank higher.

    Args:
        query_text (str): Raw query string.
        inverted_index (dict): Inverted index (term -> list of pids).
        bm25_idf (dict): Precomputed BM25 IDF scores.
        df_docs (pd.DataFrame): DataFrame indexed by 'pid', containing 'tokens', 'doc_length', 'average_rating', 'discount'.
        k (int): Number of top results to return.
        k1, b (float): BM25 hyperparameters.
        w_rating (float): Weight for average_rating boost.
        w_discount (float): Weight for discount boost.
        w_price (float): Weight for price decay.
        w_brand (float): Weight for brand average rating boost. (confidence in the brand even if the specific product has low rating or no rating)
    Returns:
        List[dict]: Ranked list of top-k results with pid, title, brand, etc
    """

    # Preprocessing
    query_tokens = preprocess_query(query_text, stemmer, stop_words)
    if not query_tokens:
        return []

    # Candidate documents
    doc_sets = [set(inverted_index[t]) for t in query_tokens if t in inverted_index]
    if not doc_sets or len(doc_sets) < len(query_tokens):
        return []
    candidate_pids = set.intersection(*doc_sets)
    if not candidate_pids:
        return []

    # Compute BM25 scores for candidate documents
    scores = {}
    L_ave = df_docs['doc_length'].mean()

    for pid in candidate_pids:
        row = df_docs.loc[pid]
        doc_tokens = row['tokens']
        Ld = row['doc_length']
        bm25_score = 0.0

        # term frequencies in this doc
        tf_doc = Counter(doc_tokens)

        for t in query_tokens:
            if t not in bm25_idf_scores:
                continue
            tf_td = tf_doc.get(t, 0)
            if tf_td == 0:
                continue

            idf = bm25_idf_scores[t]

            # BM25 term contribution
            denom = tf_td + k1 * ((1 - b) + b * (Ld / L_ave))
            term_score = idf * ((k1 + 1) * tf_td / denom)
            bm25_score += term_score
        
        # We obtain the numeric values of average_rating and discount (with 0 default if missing)
        # Convert to native Python float and handle NaN values properly
        rating_val = row.get('average_rating', 0)
        rating = float(rating_val) if pd.notna(rating_val) else 0.0
        
        discount_val = row.get('discount', 0)
        discount = float(discount_val) if pd.notna(discount_val) else 0.0
        
        price_val = row.get('selling_price', 0)
        price = float(price_val) if pd.notna(price_val) else 0.0
        
        brand = row.get('brand_facet', 'unknown')
        
        # Compute 'boosts'
        # We use np.log1p (log(1+x)), safe for values of 0

        # Boosts
        rating_boost = 1 + (w_rating * np.log1p(rating))

        discount_boost = 1 + (w_discount * np.log1p(discount))

        avg_brand_rating = brand_ratings_map.get(brand, DEFAULT_BRAND_RATING)
        # Ensure brand rating is also a native float
        avg_brand_rating = float(avg_brand_rating) if pd.notna(avg_brand_rating) else float(DEFAULT_BRAND_RATING)
        brand_boost = 1 + (w_brand * np.log1p(avg_brand_rating))

        # Decay
        price_decay = 1 / (1 + (w_price * price)/1000)


        
        # Compute final score
        final_score = bm25_score * rating_boost * discount_boost * price_decay * brand_boost

        if final_score > 0:
            scores[pid] = final_score

    if not scores:
        return []

    # --- Sort and format top-k results ---
    sorted_pids = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    results = []
    for pid, s in sorted_pids:
        row = df_docs.loc[pid]
        results.append({
            "title": row.get("title", ""),
            "score": float(s),
            "pid": pid,
            "brand": row.get("brand_facet", ""),
            "url": row.get("url", ""),
            "selling_price": row.get("selling_price", np.nan),
            "average_rating": row.get("average_rating", np.nan),
            "discount": row.get("discount", np.nan) 
        })
    return results

## 1.4 Ranking Comparisons

In [16]:
#Test query list
five_query_list = ["blue shirt round neck machine wash", "solid men blue trackpants", "full sleeve casual shirt cotton", "women polo t-shirt", "white shirt cotton"]

In [17]:
for query in five_query_list:
    print(f"TF-IDF Results for query: '{query}'")
    results_tfidf = search_tfidf(query, inverted_index, idf_scores, df_indexed, k=5)
    for res in results_tfidf:
        print(res)
    print("\n")

    print(f"BM25 Results for query: '{query}'")
    results_bm25 = search_bm25(query, inverted_index, bm25_idf_scores, df_indexed, k=5)
    for res in results_bm25:
        print(res)
    print("\n")

    print(f"Custom Results for query: '{query}'")
    results_custom = search_custom(query, inverted_index, bm25_idf_scores, df_indexed, brand_avg_ratings , k=5, w_rating=0.4, w_discount=0.1, w_price=0.1, w_brand=0.1)
    for res in results_custom:
        print(res)
    print("\n\n---------------------\n\n")

TF-IDF Results for query: 'blue shirt round neck machine wash'
{'title': 'Printed Men Round Neck Blue T-Shirt', 'score': np.float64(0.7132183545459758), 'pid': 'TSHFXHAPHNFGX68Q', 'url': 'https://www.flipkart.com/rose-wear-printed-men-round-neck-blue-t-shirt/p/itm50b46d94567ee?pid=TSHFXHAPHNFGX68Q&lid=LSTTSHFXHAPHNFGX68QVMK2VG&marketplace=FLIPKART&srno=b_3_105&otracker=browse&fm=organic&iid=b959a0e1-6898-4d40-a5ed-197450a4fb57.TSHFXHAPHNFGX68Q.SEARCH&ssid=wrfonoocow0000001612412993603', 'selling_price': np.float64(215.0), 'brand': 'rose_we'}
{'title': 'Printed Men Round Neck Blue T-Shirt', 'score': np.float64(0.7047741417621183), 'pid': 'TSHFVSS3Q8GGG3XG', 'url': 'https://www.flipkart.com/xink-printed-men-round-neck-blue-t-shirt/p/itmda60b6ec09c7e?pid=TSHFVSS3Q8GGG3XG&lid=LSTTSHFVSS3Q8GGG3XGETIAW9&marketplace=FLIPKART&srno=b_1_3&otracker=browse&fm=organic&iid=376481ae-575a-402b-93e4-c1a1f55236d3.TSHFVSS3Q8GGG3XG.SEARCH&ssid=ql2g4kd1y80000001612415491360', 'selling_price': np.float64(39

# 2. Implementing  word2vec + cosine ranking score.

##  2.1 Loading and setting up


In [18]:
# we installed gensim with:
# pip install gensim

## 2.2 Function definition to generate a text with word2vec 

In [19]:
sentences = df['processed_text'].apply(lambda x: word_tokenize(" ".join(x)))  
word2vec_model = gensim.models.Word2Vec(sentences, vector_size=300, window=5, min_count=1, workers=4)  # Train the model
word2vec_model.save("my_word2vec_model.model")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [20]:
def text_to_vector(text, word2vec_model):  #create single vector by averaging 
    
    words = word_tokenize(text)  # Tokenize the text
    vectors = []

    for word in words:
        try:
            vectors.append(word2vec_model.wv[word])  
        except KeyError:
            continue  # If the word is not in the model, we ignore it

    if vectors:
        return np.mean(vectors, axis=0)  # We return the average vector
    else:
        return np.zeros(word2vec_model.vector_size)  # zero vector if no valid words found

In [21]:
# Generate document vectors for each document
df['document_vector'] = df['processed_text'].apply(lambda x: text_to_vector(" ".join(x), word2vec_model))


## 2.3 Ranking 


In [22]:
def search_word2vec(query_text, inverted_index, df_docs, word2vec_model, k=20):
   
    query_tokens_search = preprocess_query(query_text, stemmer, stop_words) #the same cleaning we used in TF-IDF/BM25
    if not query_tokens_search:
        return []

    # sets of document IDs for each token in the query using the inverted index
    doc_sets = [set(inverted_index[t]) for t in query_tokens_search if t in inverted_index]
    if len(doc_sets) != len(query_tokens_search):
       
        return []


    # find the intersection of all document sets to get the candidate document IDs
    candidate_pids = set.intersection(*doc_sets)
    if not candidate_pids:
        return []

   
    # convert the raw query text into a vector using word2vec
    query_vector = text_to_vector(query_text.lower(), word2vec_model)

 
    scores = {} # to store the cosine similarity scores between the query and documents

    # for each candidate document, retrieve its document vector and compute cosine similarity
    for pid in candidate_pids:

        doc_vec = df_docs.loc[pid, 'document_vector']
        # doc_vec is already a numpy array from your previous step
        
        # Manual cosine similarity calculation: cos(θ) = (A · B) / (||A|| * ||B||)
        # where A · B is the dot product and ||A|| is the L2 norm (magnitude)
        dot_product = np.dot(query_vector, doc_vec)
        query_norm = np.linalg.norm(query_vector)
        doc_norm = np.linalg.norm(doc_vec)
        
        # Avoid division by zero
        if query_norm == 0 or doc_norm == 0:
            sim = 0.0
        else:
            sim = dot_product / (query_norm * doc_norm)
        
        scores[pid] = sim

    if not scores:
        return []

   
    top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k] # sorting the results 

    results = []
    for pid, score in top:
        row = df_docs.loc[pid]
        results.append({
            "pid": pid,
            "score": float(score),
            "title": row.get("title", ""),
            "brand": row.get("brand_facet", ""),
            "url": row.get("url", ""),
            "selling_price": row.get("selling_price", np.nan)
        })

    return results


In [23]:
# Set the DataFrame index by pid
df_indexed = df.set_index('pid')

# Define the queries again as in previous lab 
five_query_list = [
    "blue shirt round neck machine wash",
    "solid men blue trackpants",
    "full sleeve casual shirt cotton",
    "women polo t-shirt",
    "white shirt cotton"
]

# Iterate over each query in the list
for q in five_query_list:
    print("\n" + "="*80)
    print(f"Query: {q}")
    
    # Search using the Word2Vec model and inverted index
    results = search_word2vec(q, inverted_index, df_indexed, word2vec_model, k=20)
    
    # Check if results are empty
    if not results:
        print("No results found.")
        continue
    
    # Print the top 20 results
    for rank, r in enumerate(results, start=1):
        print(f"{rank:2d}. score={r['score']:.4f} | pid={r['pid']} | "
              f"title={r['title'][:80]} | brand={r['brand']} | price={r['selling_price']}")



Query: blue shirt round neck machine wash
 1. score=0.9078 | pid=TSHFVYHCYY5YPGZT | title=Solid Women Round Neck Dark Blue T-Shirt | brand=steenb | price=449.0
 2. score=0.8983 | pid=TSHFZ3JD8H4Z5NV7 | title=Solid Men Round Neck Blue T-Shirt | brand=reeb | price=749.0
 3. score=0.8983 | pid=TSHFZ3JEBFDR9XUE | title=Solid Men Round Neck Blue T-Shirt | brand=reeb | price=749.0
 4. score=0.8978 | pid=TSHFZF6KDGVRSZKN | title=Printed Women Round Neck Blue T-Shirt | brand=adidas_origina | price=1264.0
 5. score=0.8967 | pid=TSHFMF3NPPDCEPND | title=Printed Men Round Neck Blue T-Shirt | brand=arbo | price=426.0
 6. score=0.8967 | pid=TSHFME2EUDE7SNHV | title=Printed Men Round Neck Blue T-Shirt | brand=steenb | price=549.0
 7. score=0.8967 | pid=TSHFNV35W6XMETKT | title=Printed Men Round Neck Blue T-Shirt | brand=tee_bud | price=399.0
 8. score=0.8950 | pid=TSHFZ3JDTYFSPMW9 | title=Solid Men Round Neck Blue T-Shirt | brand=reeb | price=768.0
 9. score=0.8942 | pid=TSHFVXGQ73ZRG9AW | title=Pr

## 2.4 Doc2Vec Implementation

In [24]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

print("Preparing tagged documents for Doc2Vec training...")
tagged_documents = [TaggedDocument(words=row['processed_text'], tags=[row['pid']])
                    for _, row in df.iterrows()]

print(f"Training Doc2Vec model on {len(tagged_documents)} documents...")
doc2vec_model = Doc2Vec(
    documents=tagged_documents,
    vector_size=300,
    window=8,
    min_count=1,
    workers=4,
    epochs=40,
    dm=1,
    negative=5,
    hs=0,
    sample=1e-5
)

doc2vec_model.save("my_doc2vec_model.model")
print("Doc2Vec training complete and model saved.")


Preparing tagged documents for Doc2Vec training...
Training Doc2Vec model on 28080 documents...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Doc2Vec training complete and model saved.


In [25]:
print("Storing Doc2Vec document vectors...")
doc2vec_vectors = {pid: doc2vec_model.dv[pid] for pid in df['pid']}
df['doc2vec_vector'] = df['pid'].map(doc2vec_vectors)
print("Doc2Vec vectors stored in DataFrame (column 'doc2vec_vector').")


Storing Doc2Vec document vectors...
Doc2Vec vectors stored in DataFrame (column 'doc2vec_vector').


In [26]:
def search_doc2vec(query_text, inverted_index, df_docs, doc2vec_model, k=20, infer_epochs=40):
    """
    Ranking function that uses Doc2Vec representations and cosine similarity.
    """
    query_tokens = preprocess_query(query_text, stemmer, stop_words)
    if not query_tokens:
        return []

    doc_sets = [set(inverted_index[t]) for t in query_tokens if t in inverted_index]
    if len(doc_sets) != len(query_tokens):
        return []

    candidate_pids = set.intersection(*doc_sets)
    if not candidate_pids:
        return []

    query_vector = doc2vec_model.infer_vector(query_tokens, epochs=infer_epochs)
    query_norm = np.linalg.norm(query_vector)
    if query_norm == 0:
        return []

    scores = {}
    for pid in candidate_pids:
        doc_vec = doc2vec_model.dv[pid]
        doc_norm = np.linalg.norm(doc_vec)
        if doc_norm == 0:
            continue
        dot_product = np.dot(query_vector, doc_vec)
        sim = dot_product / (query_norm * doc_norm)
        scores[pid] = sim

    if not scores:
        return []

    top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    results = []
    for pid, score in top:
        row = df_docs.loc[pid]
        results.append({
            "pid": pid,
            "score": float(score),
            "title": row.get("title", ""),
            "brand": row.get("brand_facet", ""),
            "url": row.get("url", ""),
            "selling_price": row.get("selling_price", np.nan)
        })
    return results


In [27]:
# Preview Doc2Vec search results for the five benchmark queries
for q in five_query_list:
    print("\n" + "="*80)
    print(f"Doc2Vec results for query: {q}")
    results = search_doc2vec(q, inverted_index, df_indexed, doc2vec_model, k=10)
    if not results:
        print("No results found.")
        continue
    for rank, r in enumerate(results, start=1):
        print(f"{rank:2d}. score={r['score']:.4f} | pid={r['pid']} | title={r['title'][:80]} | brand={r['brand']} | price={r['selling_price']}")



Doc2Vec results for query: blue shirt round neck machine wash
 1. score=0.8936 | pid=TSHFW47HADYVPYGG | title=Printed Women Round Neck Blue T-Shirt | brand=yellowvib | price=799.0
 2. score=0.8860 | pid=TSHFRDQBZJYAWPKN | title=Solid Men Round Neck Dark Blue T-Shirt | brand=m7_by_metrona | price=318.0
 3. score=0.8816 | pid=TSHFJBXAAGYHFHYH | title=Printed Women Round Neck Dark Blue, White T-Shirt  (Pack of 2) | brand=yellowvib | price=1350.0
 4. score=0.8793 | pid=TSHFGXZYMERZN5EZ | title=Color Block Men Round Neck Dark Blue, Blue T-Shirt | brand=m7_by_metrona | price=613.0
 5. score=0.8732 | pid=TSHFJWZDMUXXGZMW | title=Self Design Men Round Neck Blue T-Shirt | brand=arbo | price=474.0
 6. score=0.8719 | pid=TSHFXYRESDYZFRAD | title=Color Block Men Round Neck Blue, White, Dark Blue T-Shirt | brand=wildst | price=399.0
 7. score=0.8708 | pid=TSHFV8ZKS2MMC9BA | title=Printed Men Round Neck Dark Blue T-Shirt | brand=yellowvib | price=820.0
 8. score=0.8708 | pid=TSHFME2FGXUVYBTC | titl

### 2.4.1 Word2Vec vs Doc2Vec Comparison


In [28]:
def compare_semantic_rankers(query_text, k=10):
    """Return a DataFrame comparing Word2Vec and Doc2Vec rankings for a query."""
    w2v_results = search_word2vec(query_text, inverted_index, df_indexed, word2vec_model, k=k)
    d2v_results = search_doc2vec(query_text, inverted_index, df_indexed, doc2vec_model, k=k)

    max_len = max(len(w2v_results), len(d2v_results))
    rows = []
    for rank in range(max_len):
        row = {"Rank": rank + 1}
        if rank < len(w2v_results):
            w = w2v_results[rank]
            row.update({
                "Word2Vec PID": w['pid'],
                "Word2Vec Title": w['title'],
                "Word2Vec Score": w['score']
            })
        else:
            row.update({"Word2Vec PID": None, "Word2Vec Title": None, "Word2Vec Score": None})

        if rank < len(d2v_results):
            d = d2v_results[rank]
            row.update({
                "Doc2Vec PID": d['pid'],
                "Doc2Vec Title": d['title'],
                "Doc2Vec Score": d['score']
            })
        else:
            row.update({"Doc2Vec PID": None, "Doc2Vec Title": None, "Doc2Vec Score": None})
        rows.append(row)

    return pd.DataFrame(rows)

# Example comparison for the first query
query_to_compare = five_query_list[0]
print(f"Comparing Word2Vec vs Doc2Vec for query: '{query_to_compare}'")
display(compare_semantic_rankers(query_to_compare, k=10))


Comparing Word2Vec vs Doc2Vec for query: 'blue shirt round neck machine wash'


,Rank,Word2Vec PID,Word2Vec Title,Word2Vec Score,Doc2Vec PID,Doc2Vec Title,Doc2Vec Score
0,1,TSHFVYHCYY5YPGZT,Solid Women Round Neck Dark Blue T-Shirt,0.907803,TSHFZK9TW7KF4MXQ,Color Block Women Round Neck Blue T-Shirt,0.960140
1,2,TSHFZ3JD8H4Z5NV7,Solid Men Round Neck Blue T-Shirt,0.898334,TSHFYUQRD5DHUHJX,"Printed Men Round Neck Dark Blue, White T-Shirt",0.958465
2,3,TSHFZ3JEBFDR9XUE,Solid Men Round Neck Blue T-Shirt,0.898334,TSHFUZX49UCJ7NBQ,"Color Block Women Round Neck Light Blue, White...",0.958317
3,4,TSHFZF6KDGVRSZKN,Printed Women Round Neck Blue T-Shirt,0.897770,TSHFHKZYHGQWPYHH,"Color Block Women Round Neck Blue, Black T-Shirt",0.956542
4,5,TSHFMF3NPPDCEPND,Printed Men Round Neck Blue T-Shirt,0.896684,TSHFZWRTQQGU4UYV,"Solid Men Round Neck Maroon, Blue, Light Blue,...",0.956439
5,6,TSHFME2EUDE7SNHV,Printed Men Round Neck Blue T-Shirt,0.896684,TSHFGQKMAPDWGJBT,Graphic Print Men Round Neck Dark Blue T-Shirt,0.956181
6,7,TSHFNV35W6XMETKT,Printed Men Round Neck Blue T-Shirt,0.896684,TSHFVTWVXCZYJZFV,Typography Men Round Neck Blue T-Shirt,0.955830
7,8,TSHFZ3JDTYFSPMW9,Solid Men Round Neck Blue T-Shirt,0.895013,TSHFF8Q5QH8NDZ68,Superhero Women Round Neck Blue T-Shirt,0.955600
8,9,TSHFVXGQ73ZRG9AW,Printed Women Round Neck Blue T-Shirt,0.894179,TSHFUMRVVTFBGKGH,Graphic Print Men Round Neck Light Blue T-Shirt,0.954777
9,10,TSHFZF6K4QRP24KN,Printed Men Round Neck Dark Blue T-Shirt,0.891216,TSHFYW5HS5ZEVBTG,"Printed Men Round Neck Pink, White, Dark Blue ...",0.954195


### 2.3.1 Comparing 5 ranking algorithms

In [29]:
# i. Precision@K (P@K)
def precision_at_k(y_true, y_score, k=10):
    """
    y_true : array-like, ground truth (1 = relevant, 0 = not relevant)
    y_score : array-like, predicted relevance scores
    k : int, number of top documents to consider
    """
    # sort indices by predicted score (descending)
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order[:k])
    
    # number of relevant docs in top-k
    relevant = np.sum(y_true)
    return relevant / k if k > 0 else 0.0


# ii. Recall@K (R@K)
def recall_at_k(y_true, y_score, k=10):
    """
    Computes Recall@K: fraction of all relevant documents retrieved in the top-K results.
    
    Parameters:
    - y_true: array of ground truth labels (1=relevant, 0=not relevant)
    - y_score: array of predicted relevance scores
    - k: number of top results to consider
    
    Returns:
    - Recall@K value (float)
    """
    # Sort by predicted score (descending) - maximum values first
    order = np.argsort(y_score)[::-1] #[::-1] means reverse the array (flip its order) because np.argsort does it ascending
    y_true_sorted = np.take(y_true, order)
    
    # Top-K slice
    y_topk = y_true_sorted[:k]
    
    # Count of relevant docs in top-K
    retrieved_relevant = np.sum(y_topk)
    
    # Total number of relevant docs in the whole dataset
    total_relevant = np.sum(y_true)
    
    if total_relevant == 0:
        return 0.0
    
    return float(retrieved_relevant) / total_relevant



# iii. Average Precision@K (P@K)
def avg_precision_at_k(y_true, y_score, k=10):
    """
    Approximates the area under the precision-recall curve.
    """
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order)
    num_relevant = 0
    precisions = []

    for i in range(min(k, len(y_true))):
        if y_true[i] == 1:
            num_relevant += 1
            precisions.append(num_relevant / (i + 1))

    if num_relevant == 0:
        return 0.0
    return np.mean(precisions)


# iv. F1-Score@K
def f1_at_k(y_true, y_score, k=10):
    """
    Harmonic mean of Precision@K and Recall@K.
    """
    p = precision_at_k(y_true, y_score, k)
    r = recall_at_k(y_true, y_score, k)
    return (2 * p * r) / (p + r) if (p + r) > 0 else 0.0


# vi. Mean Reciprocal Rank (MRR)
def rr_at_k(y_true, y_score, k=10):
    """
    Returns the inverse of the rank of the first relevant document.
    """
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order[:k])
    if np.sum(y_true) == 0:
        return 0.0
    first_rel = np.argmax(y_true)
    return 1 / (first_rel + 1)


def mrr_at_k(search_res, k=10):
    """
    Mean Reciprocal Rank@K across all queries.
    """
    rr_scores = []
    for q in search_res['query_id'].unique():
        curr_data = search_res[search_res['query_id'] == q]
        rr = rr_at_k(curr_data['y_true'].to_numpy(),
                     curr_data['y_score'].to_numpy(),
                     k)
        rr_scores.append(rr)
    return np.mean(rr_scores) if rr_scores else 0.0


# vii. Normalized Discounted Cumulative Gain (NDCG)
def dcg_at_k(y_true_sorted, k=10):
    """Discounted Cumulative Gain@K"""
    y_true_sorted = np.asarray(y_true_sorted, dtype=float)
    gains = 2.0 ** y_true_sorted[:k] - 1.0
    discounts = np.log2(np.arange(2, len(gains) + 2))
    return np.sum(gains / discounts)


def ndcg_at_k(y_true, y_score, k=10):
    """Normalized DCG@K"""
    # ensure numpy
    y_true = np.asarray(y_true, dtype=float)
    y_score = np.asarray(y_score, dtype=float)
    order = np.argsort(y_score)[::-1]
    rel_sorted = y_true[order]
    dcg = dcg_at_k(rel_sorted, k)

    # ideal DCG: sort true labels desc, take top-k
    ideal_rel = np.sort(y_true)[::-1]
    idcg = dcg_at_k(ideal_rel, k)

    return 0.0 if idcg == 0 else dcg / idcg



In [30]:
def evaluate_method(method_name, search_fn, five_query_list, ground_truth, k=10):
    rows = []

    for qid, query_text in enumerate(five_query_list, start=1):

        # Run the ranking function (TF-IDF, BM25, custom, Word2Vec)
        results = search_fn(query_text)

        # If empty → fill metric values with zeros
        if not results:
            rows.append({
                "Method": method_name,
                "Query": qid,
                "P@10": 0, "R@10": 0, "AP@10": 0,
                "F1@10": 0, "MRR@10": 0, "nDCG@10": 0
            })
            continue

        # Extract system scores
        y_score = np.array([r['score'] for r in results[:k]])

        # Ground truth labels for this query
        y_true  = np.array(ground_truth[qid])

        # Compute metrics
        rows.append({
            "Method": method_name,
            "Query": qid,
            "P@10":  precision_at_k(y_true, y_score, k),
            "R@10":  recall_at_k(y_true, y_score, k),
            "AP@10": avg_precision_at_k(y_true, y_score, k),
            "F1@10": f1_at_k(y_true, y_score, k),
            "MRR@10": rr_at_k(y_true, y_score, k),
            "nDCG@10": ndcg_at_k(y_true, y_score, k)
        })

    return rows


In [31]:
# Define ground truth for each query (your expert labels)
ground_truth = {
    1: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    2: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    3: [1, 1, 1, 0, 1, 1, 0, 0, 0, 0],
    4: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    5: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

In [32]:
eval_rows = []

eval_rows += evaluate_method(
    "TF-IDF",
    lambda q: search_tfidf(q, inverted_index, idf_scores, df_indexed, k=10),
    five_query_list,
    ground_truth
)

eval_rows += evaluate_method(
    "BM25",
    lambda q: search_bm25(q, inverted_index, bm25_idf_scores, df_indexed, k=10),
    five_query_list,
    ground_truth
)

eval_rows += evaluate_method(
    "CustomScore",
    lambda q: search_custom(q, inverted_index, bm25_idf_scores, df_indexed, brand_avg_ratings, k=10),
    five_query_list,
    ground_truth
)

eval_rows += evaluate_method(
    "Word2Vec",
    lambda q: search_word2vec(q, inverted_index, df_indexed, word2vec_model, k=10),
    five_query_list,
    ground_truth
)

eval_rows += evaluate_method(
    "Doc2Vec",
    lambda q: search_doc2vec(q, inverted_index, df_indexed, doc2vec_model, k=10),
    five_query_list,
    ground_truth
)


In [33]:
comparison_df = pd.DataFrame(eval_rows)
comparison_df = comparison_df.round(3)
comparison_df

,Method,Query,P@10,R@10,AP@10,F1@10,MRR@10,nDCG@10
0,TF-IDF,1,1.0,1.0,1.000,1.000,1.0,1.000
1,TF-IDF,2,0.9,1.0,1.000,0.947,1.0,1.000
2,TF-IDF,3,0.5,1.0,0.927,0.667,1.0,0.975
3,TF-IDF,4,1.0,1.0,1.000,1.000,1.0,1.000
4,TF-IDF,5,0.0,0.0,0.000,0.000,0.0,0.000
5,BM25,1,1.0,1.0,1.000,1.000,1.0,1.000
6,BM25,2,0.9,1.0,1.000,0.947,1.0,1.000
7,BM25,3,0.5,1.0,0.927,0.667,1.0,0.975
8,BM25,4,1.0,1.0,1.000,1.000,1.0,1.000
9,BM25,5,0.0,0.0,0.000,0.000,0.0,0.000
